In [ ]:
import os
from ultralytics import YOLO
import torch

def setup_dataset():
    """
    Authenticates with Roboflow and streams the zipped imagery arrays
    directly into the remote Colab scratch disk environment.
    """
    # Enforce standard dataset root structure
    target_dir = "/content/datasets/ppe_data"
    os.makedirs(target_dir, exist_ok=True)

    # Check if data already exists to avoid redundant downloads on re-runs
    if os.path.exists(os.path.join(target_dir, "data.yaml")):
        print("✅ Dataset already present. Skipping download step.")
        return

    print("🛰️ Ingesting dataset from Roboflow Universe...")
    try:
        from roboflow import Roboflow

        # paste your exact snippet copied from Roboflow here:
        rf = Roboflow(api_key="YOUR_PRIVATE_API_KEY")
        project = rf.workspace("workspace-name").project("construction-site-safety")
        version = project.version(1)

        # Enforce downloading in YOLO format directly to our target directory
        dataset = version.download("yolov8", location=target_dir)
        print(f"✅ Data successfully isolated at: {target_dir}")

    except Exception as e:
        print(f"❌ Dataset ingestion failed: {e}")
        raise

def main():
    # Verify cloud GPU accessibility via PyCharm interpreter mapping
    device = 0 if torch.cuda.is_available() else "cpu"
    print(f"🚀 Execution routing to Remote Device ID: {device}")

    # 1. Fetch and unpack data assets
    setup_dataset()

    # 2. Instantiate pre-trained lightweight nano backbone
    model = YOLO("yolov8n.pt")

    # 3. Spin up cloud transfer learning loop
    model.train(
        data="config/data.yaml",   # Path relative to your PyCharm project root
        epochs=15,                 # Quick verification epoch window
        imgsz=640,
        batch=32,                  # T4 GPU optimized batch sizing
        device=device,
        workers=2,
        plots=True
    )

if __name__ == "__main__":
    main()